# Week 5 Practical: Data Wrangling
### Data Frame Ecosystems + Building a Feature Table — PUBH 6854 / PUBH 4201
Sep 21 & Sep 23, 2026

This one notebook covers both of this week's lecture days: **Part 1** is the
pandas-vs-polars side of Day 1's data-frame-ecosystems live demo (the
tidyverse side lives in `Rmarkdown/week05_practical.Rmd` — pick whichever
language you're using for Lab 3, you don't need both), and **Part 2** is Day
2's samples × features × metadata hands-on exercise.

**Data:** `../gene_counts_long.csv`, `../sample_metadata.csv`,
`../gene_metadata.csv` — a small synthetic gene-expression-style dataset
built for this exercise (see `../SOURCE.md`). Deliberately **not** the Lab 3
dataset — different shape, different gene panel, and clean-but-incomplete
rather than messy, since the point here is the pivot/join/readiness
workflow, not regex-style text parsing.

## Setup

If you don't already have `polars` in your `notebooks` conda environment
(most of you won't yet — it's new this week), install it once:

```console
$ conda activate notebooks
$ pip install polars
```

See `../Jupyter/environment.yml` if you need to rebuild the environment
from scratch.

In [ ]:
import pandas as pd
import polars as pl

# pandas
counts_pd = pd.read_csv("../gene_counts_long.csv")
samples_pd = pd.read_csv("../sample_metadata.csv")
genes_pd = pd.read_csv("../gene_metadata.csv")

# polars
counts_pl = pl.read_csv("../gene_counts_long.csv")
samples_pl = pl.read_csv("../sample_metadata.csv")
genes_pl = pl.read_csv("../gene_metadata.csv")

print(f"{len(counts_pd)} count rows, {len(samples_pd)} samples, {len(genes_pd)} genes")
counts_pd.head()

## Part 1 — Data frame ecosystems: the same pipeline, three ways

Monday's lecture walked through **filter → select → mutate/assign →
group-by/summarize → join** side by side in tidyverse, pandas, and polars.
Below is the pandas/polars half of that, using our actual data instead of
the lecture's toy example. The tidyverse version of every cell here has a
line-for-line equivalent in `Rmarkdown/week05_practical.Rmd` — worth
skimming even if R isn't your language this semester, just to see how the
same concepts map across ecosystems.

### Filter

Keep only samples in the `treatment` group.

In [ ]:
# pandas
treatment_pd = samples_pd[samples_pd["treatment_group"] == "treatment"]
print("pandas:", len(treatment_pd), "rows")
treatment_pd.head(3)

In [ ]:
# polars
treatment_pl = samples_pl.filter(pl.col("treatment_group") == "treatment")
print("polars:", treatment_pl.height, "rows")
treatment_pl.head(3)

### Select

Keep only the columns we need for a quick site/group summary.

In [ ]:
# pandas
cols_pd = samples_pd[["sample_id", "site", "treatment_group"]]
cols_pd.head(3)

In [ ]:
# polars
cols_pl = samples_pl.select(["sample_id", "site", "treatment_group"])
cols_pl.head(3)

### Mutate / assign

Add a derived column — `log1p` of the raw count, a common transform for
count data before modeling (compresses the right skew from a handful of
very highly expressed genes like `GAPDH`/`ACTB`).

In [ ]:
import numpy as np

# pandas
counts_pd["log_count"] = np.log1p(counts_pd["raw_count"])
counts_pd.head(3)

In [ ]:
# polars
counts_pl = counts_pl.with_columns(
    (pl.col("raw_count") + 1).log().alias("log_count")
)
counts_pl.head(3)

### Group-by / summarize

Mean raw count per gene, across all samples — a sanity check that
housekeeping genes (`GAPDH`, `ACTB`) really are the highest-expressed, as
`../gene_metadata.csv`'s `gene_class` column claims.

In [ ]:
# pandas
summary_pd = (
    counts_pd.groupby("gene_id")["raw_count"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
)
summary_pd

In [ ]:
# polars
summary_pl = (
    counts_pl.group_by("gene_id")
    .agg(pl.col("raw_count").mean().round(1).alias("mean_count"))
    .sort("mean_count", descending=True)
)
summary_pl

### Join

Attach sample-level metadata (site, treatment group, age) onto the
long-format counts table — this is the join that sets up Part 2's feature
table.

In [ ]:
# pandas
merged_pd = counts_pd.merge(samples_pd, on="sample_id", how="left")
merged_pd.head(3)

In [ ]:
# polars
merged_pl = counts_pl.join(samples_pl, on="sample_id", how="left")
merged_pl.head(3)

**Checkpoint:** `merged_pd` and `merged_pl` should have the same number of
rows as `counts_pd`/`counts_pl` (a left join keeps every count row, even
if — hypothetically — a sample_id had no metadata match) and the same
number of columns as `counts` + `samples` minus the shared `sample_id`
key. If your row count changed, that's usually a sign of a many-to-many
join somewhere — worth tracking down rather than ignoring.

## Wide vs. long / tidy data

**Tidy data principle:** one observation per row, one variable per column.
`gene_counts_long.csv` is tidy *for a "count" observation* (each row is one
sample-gene measurement) — but it's the wrong shape for "one row per
sample," which is what most downstream modeling and plotting tools expect.
Getting from one to the other is a **pivot** (long → wide) or its inverse
(wide → long), not a join — no new information is added, only reshaped.

Below: pivot `counts` from long (one row per sample×gene) to wide (one row
per sample, one column per gene) — the samples × features shape Part 2
builds on.

In [ ]:
# pandas: pivot_table (handles the missing sample-gene pairs as NaN automatically)
wide_pd = counts_pd.pivot_table(index="sample_id", columns="gene_id", values="raw_count")
wide_pd

In [ ]:
# polars: pivot
wide_pl = counts_pl.pivot(index="sample_id", columns="gene_id", values="raw_count")
wide_pl

Notice the `NaN`/`null` cells — those are the ~6% of sample×gene pairs
that were never in `gene_counts_long.csv` to begin with (see `../SOURCE.md`).
The pivot didn't create this missingness, it just made it *visible* as
actual cells instead of absent rows. That's exactly the kind of thing
"analytic data readiness" means catching before you hand a table to a
model — see Part 2.

## Part 2 — Constructing a samples × features × metadata table (Day 2)

**The canonical shape:** one row per sample, one column per feature
(here, genes), plus sample-level metadata columns (age, sex, site,
treatment_group) either alongside the feature columns or joinable by
`sample_id`. This is the shape nearly every downstream analysis —
statistical modeling, ML, visualization — expects.

**Recap of the pieces:**
- `gene_counts_long.csv` — raw pipeline export, long format (what Part 1
  pivoted above)
- `sample_metadata.csv` — one row per sample, clinical/demographic fields
- `gene_metadata.csv` — one row per gene, feature-level annotation

### Your turn: build the full feature table

Using `wide_pd` (or `wide_pl`) from above, join in `sample_metadata.csv` so
every row has both the gene columns *and* the sample metadata. Fill in the
`TODO` below.

In [ ]:
# TODO: reset the index of wide_pd so sample_id is a real column again,
# then merge samples_pd onto it (on="sample_id", how="left")
feature_table = wide_pd.reset_index().merge(samples_pd, on="sample_id", how="left")
feature_table

### Data readiness check

Before this table goes anywhere near a model, run through a short
checklist — this is what "analytic data readiness" means concretely, not
just an abstract idea:

1. **Consistent types** — is every gene column numeric? Is `age` numeric
   (watch out: a blank cell can silently make a whole column `object`/
   string in pandas)?
2. **Documented missingness** — how many missing values, where, and do you
   know *why* (dropped-out gene measurement vs. a metadata field nobody
   filled in are different problems with different fixes)?
3. **No leakage between metadata and features** — `treatment_group` is
   useful to know, but would using it as a *feature* in a model that's
   trying to predict treatment response be circular? (No single right
   answer — the point is to ask.)
4. **Units resolved** — not an issue for raw counts here, but always check
   for a real dataset (see Lab 3's `messy_samples.csv` for what it looks
   like when this *isn't* resolved).

Run the missingness check below, then answer the two questions in the
markdown cell after it.

In [ ]:
# TODO: count missing values per column in feature_table
feature_table.isna().sum()

**Answer these (no need to write code — just think it through, same as
the week's discussion prompt):**

1. Which column(s) have the most missing values, and does that match what
   `../SOURCE.md` told you to expect?
2. `age` is missing for one sample. Before you decide how to handle it
   (drop the row? impute the mean? leave it and let downstream tools
   handle `NaN`?) — what would you need to know first, and who would you
   ask? (This is this week's discussion prompt, applied to your own
   table.)

### Optional extension: bring in feature-level metadata

`gene_metadata.csv` has a `gene_class` column (`housekeeping`, `oncogene`,
`cytokine`, ...). Try filtering `feature_table` down to just the cytokine
genes (`IL6`, `TNF`) using `genes_pd` to look up which columns those are —
a small taste of how feature *and* sample metadata both matter when you're
deciding what to include in an analysis.

In [ ]:
# TODO (optional): filter genes_pd to gene_class == "cytokine", get the
# gene_id list, and select just those columns (+ sample_id) from feature_table
cytokine_genes = genes_pd.loc[genes_pd["gene_class"] == "cytokine", "gene_id"].tolist()
feature_table[["sample_id"] + cytokine_genes]

## Wrap-up

| | |
|---|---|
| **Due Wed, Sep 30, 11:59pm** | Lab 3: Parsing Messy Health or Genomic Data |
| **Keep for later** | This "samples × features × metadata" shape comes back directly in Week 8-9 (Lab 4: Database-Driven Feature Table) — same framing, different (relational-database) source data |
| **Closes Module 1** | Environments (Wk1-2) → notebooks (Wk3) → text parsing (Wk4) → structured tables (Wk5) — Module 2 (Wk6+) shifts to software design |